# M8_8.22–M8_8.24 · NumPy, pandas y Matplotlib

Los ejemplos usan archivos reales del repositorio: CSV, Excel y SQLite. Los tres formatos contienen las mismas tablas lógicas para comparar flujos de lectura.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_NAME = "m8-herramientas-data-science"
GITHUB_USER = "REPLACE_WITH_YOUR_GITHUB_USERNAME"  # El profesorado lo cambia una vez antes de publicar

def localizar_repo():
    actual = Path.cwd().resolve()
    for candidato in [actual, *actual.parents]:
        if (candidato / "data" / "input").exists():
            return candidato
    if "google.colab" in sys.modules:
        destino = Path("/content") / REPO_NAME
        if not destino.exists():
            url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
            if GITHUB_USER.startswith("REPLACE_"):
                raise RuntimeError("El profesorado debe configurar GITHUB_USER antes de publicar el repositorio.")
            subprocess.run(["git", "clone", "--depth", "1", url, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encuentra la raiz del repositorio.")

ROOT = localizar_repo()
DATA = ROOT / "data" / "input"
DB = ROOT / "data" / "database" / "hidrogeologia.sqlite"
OUTPUT = ROOT / "data" / "output"
for carpeta in [OUTPUT / "tables", OUTPUT / "figures", OUTPUT / "logs"]:
    carpeta.mkdir(parents=True, exist_ok=True)
print("Repositorio:", ROOT)

# M8_8.22 · NumPy

Un array es una estructura homogénea de varias dimensiones. `shape` describe dimensiones; `dtype`, tipo; `axis`, dirección de una operación. Las máscaras booleanas seleccionan valores.

In [ ]:
import numpy as np
niveles=np.array([8.7,8.4,np.nan,9.1])
print(niveles.shape,niveles.dtype)
print("Media valida:",np.nanmean(niveles))
print("Mayores de 8.5:",niveles[niveles>8.5])

## Vectorización y ejes

Las operaciones vectorizadas actúan sobre arrays completos. En una matriz, `axis=0` resume columnas y `axis=1` resume filas.

In [ ]:
matriz=np.array([[8.1,8.4,8.6],[14.2,14.0,13.8]])
print("Por columna:",matriz.mean(axis=0))
print("Por fila:",matriz.mean(axis=1))

# M8_8.23 · pandas y fuentes de datos

Un DataFrame representa una tabla. En la práctica, los datos llegan en archivos o bases de datos. Primero se inspeccionan columnas, tipos, fechas, unidades y claves.

In [ ]:
import pandas as pd, sqlite3
pozos_csv=pd.read_csv(DATA/"pozos.csv")
med_csv=pd.read_csv(DATA/"mediciones.csv",parse_dates=["fecha"])
display(pozos_csv.head(),med_csv.head())
print(med_csv.dtypes)

## Importar Excel

`read_excel` permite seleccionar una hoja. El formato Excel puede ser útil para intercambio, pero requiere comprobar nombres de hoja, tipos y fechas.

In [ ]:
pozos_xlsx=pd.read_excel(DATA/"datos_hidrogeologicos.xlsx",sheet_name="pozos")
med_xlsx=pd.read_excel(DATA/"datos_hidrogeologicos.xlsx",sheet_name="mediciones",parse_dates=["fecha"])
print(pozos_xlsx.shape,med_xlsx.shape)

## Importar desde SQLite

`read_sql_query` devuelve un DataFrame. SQL selecciona datos en la base; pandas continúa el análisis. Este patrón conecta bases de datos y análisis científico.

In [ ]:
with sqlite3.connect(DB) as con:
    med_sql=pd.read_sql_query("SELECT * FROM mediciones",con,parse_dates=["fecha"])
    combinado=pd.read_sql_query("""SELECT m.*, p.acuifero, p.cota_terreno_m
    FROM mediciones m JOIN pozos p ON m.id_pozo=p.id_pozo""",con,parse_dates=["fecha"])
print(med_sql.shape); display(combinado.head())

## Verificar equivalencia entre fuentes

Mismos datos no garantiza mismos tipos. Conviene comparar dimensiones, columnas y algunas filas, no solo asumir que CSV, Excel y SQLite producen objetos idénticos.

In [ ]:
print("CSV",med_csv.shape,"Excel",med_xlsx.shape,"SQLite",med_sql.shape)
print("Mismas columnas:",list(med_csv.columns)==list(med_sql.columns))

## Seleccionar, filtrar, agrupar y combinar

La selección debe responder a una pregunta. `groupby` resume grupos; `merge` relaciona tablas mediante una clave estable.

In [ ]:
aluvial=med_csv.loc[med_csv.id_pozo.isin(["P01","P02","P03"]),["id_pozo","fecha","profundidad_nivel_m"]]
resumen=combinado.groupby("acuifero").profundidad_nivel_m.agg(["count","mean","median","std"])
display(aluvial.head(),resumen)

## Fechas, formato largo y ancho

El formato largo facilita agrupaciones. `pivot` puede crear una tabla ancha para comparar series. Las fechas deben ser `datetime`.

In [ ]:
ancho=med_csv.pivot_table(index="fecha",columns="id_pozo",values="profundidad_nivel_m",aggfunc="first")
display(ancho.head())

# M8_8.24 · Matplotlib

Una figura debe responder a una pregunta y declarar variable, unidades, tiempo y grupos. Una figura atractiva no corrige datos incorrectos.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,4))
for pozo,g in med_csv.groupby("id_pozo"):
    ax.plot(g.fecha,g.profundidad_nivel_m,marker="o",label=pozo)
ax.invert_yaxis(); ax.set(xlabel="Fecha",ylabel="Profundidad del nivel (m)",title="Evolucion del nivel")
ax.legend(ncol=4,title="Pozo"); fig.autofmt_xdate(); plt.show()

## Dispersión e histograma

El scatter plot explora relaciones; el histograma explora una distribución. Ninguno demuestra por sí solo causalidad ni identifica automáticamente errores.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(med_csv.precipitacion_mm,med_csv.profundidad_nivel_m,alpha=.6)
axes[0].set(xlabel="Precipitacion (mm)",ylabel="Profundidad (m)",title="Precipitacion y nivel")
axes[1].hist(med_csv.profundidad_nivel_m.dropna(),bins=12,edgecolor="black")
axes[1].set(xlabel="Profundidad (m)",ylabel="Observaciones",title="Distribucion")
plt.tight_layout(); plt.show()

## Exportar resultados

Los datos originales no se sobrescriben. Las tablas y figuras derivadas se guardan en `data/output`.

In [ ]:
resumen.to_csv(OUTPUT/"tables/resumen_por_acuifero.csv")
print("Guardado:",OUTPUT/"tables/resumen_por_acuifero.csv")

## Actividad

1. Lee `mediciones.csv`.
2. Lee la misma tabla desde SQLite.
3. Comprueba dimensiones y columnas.
4. Selecciona un pozo y resume su serie trimestral.
5. Genera una figura con unidades.
6. Guarda una tabla en `data/output/tables`.